# Rainbow DQN sur CartPole-v1 : comprendre chaque piece avant l'assemblage

**Algorithme** : Rainbow DQN, version pedagogique from scratch  
**Environnement** : CartPole-v1  
**Objectif du notebook** : comprendre pourquoi Rainbow existe, ce que chaque brique corrige dans DQN, puis assembler ces briques dans un agent lisible.

Dans le notebook precedent, DQN a remplace la table Q par un reseau, puis Double DQN a corrige une partie du biais d'overestimation en separant la selection et l'evaluation de l'action suivante. Rainbow part de cette base et ajoute plusieurs ameliorations compatibles.

La phrase importante est : **Rainbow n'est pas une seule idee**. C'est une composition de corrections. Chaque correction vise une faiblesse differente du DQN classique : architecture trop monolithique, replay uniforme, propagation lente des rewards, cible scalaire pauvre, exploration trop brute.

Le fil conducteur sera donc :

```text
DQN -> Double DQN -> Dueling -> PER -> n-step -> C51 -> Noisy Nets -> Rainbow
```

On garde CartPole volontairement. Rainbow est presque trop puissant pour cet environnement, mais cette simplicite est utile : elle laisse voir les mecanismes sans se noyer dans un benchmark lourd.


## 1. Rappel : ce que DQN sait deja faire, et ce qui reste fragile

DQN apprend une fonction de valeur d'action. Pour un etat $s$, le reseau produit une valeur par action discrete. La politique gloutonne choisit ensuite l'action avec la plus grande valeur.

$$Q_{\theta}(s) = \left[Q_{\theta}(s,a_0), Q_{\theta}(s,a_1), \dots \right]$$

La cible DQN classique est un bootstrap : on prend la recompense immediate, puis on ajoute une estimation de la meilleure valeur future.

$$y = r + \gamma (1-d) \max_{a'} Q_{\theta^-}(s',a')$$

Chaque terme a un role precis : $r$ est le signal immediat, $\gamma$ devalue le futur, $d$ coupe le bootstrap quand l'episode est vraiment termine, et $\theta^-$ indique le target network, plus lent que le reseau online.

Double DQN garde cette cible, mais evite que le meme reseau choisisse et evalue l'action suivante. Le reseau online choisit l'action, le target network l'evalue :

$$a^* = \arg\max_{a'} Q_\theta(s',a'), \qquad y = r + \gamma(1-d)Q_{\theta^-}(s',a^*)$$

Rainbow conserve cette correction. Le reste du notebook demande : **quelles autres faiblesses restent apres Double DQN ?**

On va les traiter une par une.


In [ ]:
import copy
import random
from collections import deque
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. CartPole-v1 : un environnement simple pour isoler les idees

CartPole demande de pousser un chariot a gauche ou a droite pour garder une tige verticale. L'environnement est petit, dense en reward, et discret en actions. C'est exactement ce qu'il faut pour inspecter les briques de Rainbow.

| Element | Valeur | Pourquoi c'est utile ici |
|---|---:|---|
| Observation | 4 reels | assez simple pour visualiser les etats |
| Actions | 2 | DQN peut enumerer les actions |
| Reward | +1 par pas | le signal est dense et facile a lire |
| Episode reussi | survie longue | la courbe reward est interpretable |

CartPole ne prouve pas que Rainbow domine DQN dans le monde reel. Il sert plutot de microscope. Si une brique est utile, on doit pouvoir expliquer son effet avec des exemples controles avant de lancer un training.

On commence par regarder un episode aleatoire : pas pour apprendre, mais pour savoir ce que mesurent les axes.


In [ ]:
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)
env.action_space.seed(42)

print("Observation space:", env.observation_space)
print("Action space     :", env.action_space)
print("Action meanings  : 0 = push left, 1 = push right")
print("Initial obs      :", np.round(obs, 3))

angles = []
cumulative_rewards = []
total = 0.0
obs, _ = env.reset(seed=42)
for t in range(80):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total += reward
    angles.append(obs[2])
    cumulative_rewards.append(total)
    if terminated or truncated:
        break

env.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cumulative_rewards, linewidth=2, color="steelblue")
axes[0].set_title("Reward cumulee pendant un episode random")
axes[0].set_xlabel("Pas de temps")
axes[0].set_ylabel("Reward cumulee")

axes[1].plot(angles, linewidth=2, color="tomato")
axes[1].axhline(0, color="black", linewidth=1, linestyle="--")
axes[1].set_title("Angle de la tige")
axes[1].set_xlabel("Pas de temps")
axes[1].set_ylabel("Angle (radians)")

plt.tight_layout()
plt.show()
print(f"Episode random termine apres {len(cumulative_rewards)} pas, reward={total:.1f}")


## 3. Dueling network : separer la valeur de l'etat et le choix de l'action

Dans DQN, le reseau apprend directement une valeur $Q(s,a)$ pour chaque action. Autrement dit, pour chaque etat, il doit repondre en une seule fois a deux questions differentes :

1. **Est-ce que cet etat est bon ?**
2. **Quelle action est la meilleure dans cet etat ?**

Ces deux questions sont liees, mais elles ne portent pas sur la meme chose.

Sur CartPole, imaginons que la tige soit presque droite, que le chariot soit au centre, et que les vitesses soient faibles. Cet etat est globalement bon. Dans ce cas, pousser legerement a gauche ou legerement a droite peut avoir des valeurs assez proches. Le plus important est d'abord de comprendre : *je suis dans une bonne situation*.

A l'inverse, si la tige tombe vite vers la droite, l'etat est dangereux. Cette fois, le choix de l'action devient beaucoup plus important : une action peut corriger la chute, l'autre peut l'aggraver. Le reseau doit donc comprendre a la fois la qualite globale de l'etat et la preference relative entre actions.

Le reseau DQN classique ne separe pas ces deux roles. Il produit directement :

$$Q(s,a_0), Q(s,a_1), \dots, Q(s,a_k)$$

Le probleme est que chaque action doit reapprendre une partie de la meme information : si un etat est globalement bon, cette information est utile pour toutes les actions. Le dueling network rend cette structure explicite.

Au lieu de predire directement $Q(s,a)$, on demande au reseau de predire deux objets :

- une valeur d'etat $V(s)$ ;
- un avantage d'action $A(s,a)$.

Puis on reconstruit $Q(s,a)$ a partir des deux :

$$Q(s,a) = V(s) + A(s,a) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s,a')$$

Lecture terme par terme :

- $V(s)$ mesure la qualite generale de l'etat, independamment de l'action ;
- $A(s,a)$ mesure a quel point l'action $a$ est meilleure ou pire que les autres dans cet etat ;
- $\frac{1}{|\mathcal{A}|}\sum_{a'} A(s,a')$ est la moyenne des avantages sur toutes les actions ;
- on retire cette moyenne pour que les avantages soient relatifs, centres autour de zero.

Pourquoi retirer la moyenne ? Parce que sinon la decomposition n'est pas unique. Par exemple, si on ajoute $+10$ a $V(s)$ et qu'on retire $10$ a tous les $A(s,a)$, la somme $V(s)+A(s,a)$ donne exactement le meme $Q(s,a)$. Le reseau aurait donc plusieurs facons equivalentes de representer la meme chose. En centrant les avantages, on force $V(s)$ a porter la valeur commune de l'etat, et $A(s,a)$ a porter seulement la difference relative entre actions.

L'intuition a retenir :

- $V(s)$ repond a : **"suis-je dans une bonne situation ?"**
- $A(s,a)$ repond a : **"dans cet etat, quelle action se distingue ?"**
- $Q(s,a)$ combine les deux pour prendre une decision.

On ne remplace donc pas l'objectif de DQN. On apprend toujours des valeurs $Q(s,a)$ et on choisit toujours l'action avec le plus grand $Q$. La difference est seulement dans l'architecture interne du reseau : au lieu de produire $Q$ directement, il passe par une decomposition plus informative.

Rainbow garde cette structure parce qu'elle aide le reseau a generaliser entre actions. Quand la valeur de l'etat est ce qui compte le plus, $V(s)$ peut l'apprendre une seule fois. Quand le choix d'action devient critique, $A(s,a)$ peut se concentrer sur la preference relative.


In [ ]:
class DuelingNetwork(nn.Module):
    """Petit reseau dueling: trunk partage, tete V(s), tete A(s,a)."""

    def __init__(self, obs_dim, n_actions, hidden_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.value = nn.Linear(hidden_dim, 1)
        self.advantage = nn.Linear(hidden_dim, n_actions)

    def forward(self, obs):
        h = self.features(obs)
        value = self.value(h)
        advantage = self.advantage(h)
        q = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q, value, advantage


dueling_net = DuelingNetwork(obs_dim=4, n_actions=2)
probe = torch.randn(5, 4)
q, value, advantage = dueling_net(probe)
print("Q shape        :", tuple(q.shape))
print("Value shape    :", tuple(value.shape))
print("Advantage shape:", tuple(advantage.shape))
print("Exemple Q[0]   :", q[0].detach().numpy().round(3))


In [ ]:
# Exemple controle: on construit V et A a la main pour voir leurs roles.
states = ["stable", "bon mais ambigu", "danger gauche", "danger droite"]
V = np.array([8.0, 7.5, 2.0, 2.0])
A_left = np.array([0.05, -0.10, 1.8, -1.6])
A_right = np.array([-0.05, 0.10, -1.8, 1.6])
A = np.stack([A_left, A_right], axis=1)
Q = V[:, None] + A - A.mean(axis=1, keepdims=True)

x = np.arange(len(states))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(x, V, color="steelblue")
axes[0].set_title("V(s): qualite globale de l'etat")
axes[0].set_xticks(x, states, rotation=20)
axes[0].set_ylabel("Valeur")

axes[1].bar(x - 0.17, A[:, 0], width=0.34, label="gauche", color="seagreen")
axes[1].bar(x + 0.17, A[:, 1], width=0.34, label="droite", color="tomato")
axes[1].set_title("A(s,a): preference relative")
axes[1].set_xticks(x, states, rotation=20)
axes[1].legend()

axes[2].bar(x - 0.17, Q[:, 0], width=0.34, label="Q gauche", color="seagreen")
axes[2].bar(x + 0.17, Q[:, 1], width=0.34, label="Q droite", color="tomato")
axes[2].set_title("Q(s,a): recombinaison")
axes[2].set_xticks(x, states, rotation=20)
axes[2].legend()

plt.tight_layout()
plt.show()


### Interpretation du plot dueling

Le premier panneau montre que les deux premiers etats sont globalement bons : $\text{V}(s)$ est haut. Pourtant, dans $\text{bon mais ambigu}$, les avantages restent proches de zero : le reseau peut apprendre que l'etat est bon sans inventer une grande difference entre gauche et droite.

Les deux derniers etats sont globalement plus dangereux : $\text{V}(s)$ est bas. Cette fois, le panneau des avantages devient important. Dans $\text{danger gauche}$, l'action gauche recoit un bonus relatif ; dans $\text{danger droite}$, c'est l'inverse.

Ce plot montre le benefice du dueling : la valeur globale et le choix fin de l'action ne sont plus forces dans une seule tete. La prochaine faiblesse concerne non pas le reseau, mais les donnees que l'on choisit de revisiter dans le replay buffer.


## 4. Prioritized Experience Replay : revisiter plus souvent ce qui surprend le reseau

Dans DQN, on stocke les transitions dans un replay buffer :

$$
(s_t, a_t, r_t, s_{t+1}, d_t)
$$

Puis, au moment d'apprendre, on tire un mini-batch de transitions dans ce buffer. Dans un replay buffer uniforme, toutes les transitions ont la meme probabilite d'etre tirees. C'est simple et relativement sain, parce que le batch ressemble bien aux donnees collectees.

Mais ce n'est pas toujours efficace.

Certaines transitions sont deja tres bien comprises par le reseau. Si le reseau predit presque exactement leur cible Bellman, les revoir encore et encore apporte peu. D'autres transitions produisent une grosse erreur : le reseau pensait qu'une action avait une certaine valeur, mais la cible lui dit autre chose. Ces transitions sont plus informatives.

L'idee de Prioritized Experience Replay est donc :

> au lieu de revisiter toutes les transitions aussi souvent, revisiter plus souvent celles qui surprennent le reseau. c'est a dire, au lieux de refaire toujours les meme exercices que je connais deja, je me concentre sur les exercices qui me font faire des erreurs.

Pour chaque transition $i$ dans le buffer, on garde une priorite $p_i$. Cette priorite est souvent basee sur l'erreur TD :

$$
p_i = |\delta_i| + \varepsilon
$$

avec :

- $\delta_i$ : erreur TD de la transition $i$ ;
- $|\delta_i|$ : magnitude de l'erreur, donc niveau de surprise ;
- $\varepsilon$ : petit terme positif pour eviter qu'une transition ait une probabilite nulle.

Dans DQN classique, l'erreur TD ressemble a :

$$
\delta_i = y_i - Q_\theta(s_i, a_i)
$$

ou $y_i$ est la cible Bellman. Dans Rainbow avec C51, on n'a plus seulement une cible scalaire, donc on peut utiliser une quantite proche de la loss distributionnelle comme signal de priorite. L'intuition reste la meme : **plus la transition est mal predite, plus elle devient prioritaire**.

Ensuite, au lieu de tirer uniformement, on transforme les priorites en probabilites d'echantillonnage :

$$
P(i) = \frac{p_i^\alpha}{\sum_j p_j^\alpha}
$$

Lecture terme par terme :

- $P(i)$ est la probabilite de tirer la transition $i$ ;
- $p_i$ est sa priorite ;
- $\alpha$ controle l'intensite de la priorisation ;
- le denominateur $\sum_j p_j^\alpha$ normalise pour que toutes les probabilites somment a 1.

Le role de $\alpha$ est important :

- si $\alpha = 0$, alors $p_i^\alpha = 1$ pour toutes les transitions, donc on retrouve le replay uniforme ;
- si $\alpha = 1$, on suit directement les priorites ;
- si $0 < \alpha < 1$, on favorise les transitions importantes, mais sans oublier completement les autres.

Concretement, dans la boucle de training, PER intervient au moment du sampling :

```text
1. collecter une transition
2. l'ajouter au replay buffer avec une priorite initiale elevee
3. quand on apprend :
   - calculer P(i) pour chaque transition du buffer
   - tirer un batch selon ces probabilites
   - faire l'update sur ce batch
   - recalculer les erreurs/losses des transitions du batch
   - mettre a jour leurs priorites p_i
```

Il y a donc deux moments importants :

1. **Quand une transition entre dans le buffer**  
   On lui donne souvent la priorite maximale courante. Pourquoi ? Parce qu'une transition nouvelle n'a pas encore ete apprise. On veut donc lui laisser une chance d'etre vue rapidement.

2. **Apres une update**  
   On mesure a quel point chaque transition du batch etait mal predite. Puis on met a jour sa priorite. Si le reseau la comprend maintenant mieux, sa priorite baisse. Si elle reste difficile, elle continuera a revenir souvent.

Mais PER introduit un probleme : le batch n'est plus un echantillon uniforme du buffer. On a volontairement sur-represente certaines transitions. Si on applique la loss normalement, ces transitions auraient trop d'influence sur le gradient.

On corrige ce biais avec des poids d'importance :

$$
w_i = \left(\frac{1}{N P(i)}\right)^\beta
$$

Lecture terme par terme :

- $N$ est le nombre de transitions dans le buffer ;
- $P(i)$ est la probabilite avec laquelle la transition $i$ a ete tiree ;
- $N P(i)$ compare cette probabilite a une probabilite uniforme ;
- si une transition est tiree trop souvent, $P(i)$ est grand, donc $w_i$ devient plus petit ;
- $\beta$ controle la force de cette correction.

En pratique, on normalise souvent les poids dans le batch pour eviter des gradients trop grands :

$$
w_i \leftarrow \frac{w_i}{\max_j w_j}
$$

Ces poids s'integrent directement dans la loss. Au lieu de moyenner toutes les pertes de la meme facon :

$$
\mathcal{L} = \frac{1}{B}\sum_i \ell_i
$$

on pondere chaque perte individuelle :

$$
\mathcal{L} = \frac{1}{B}\sum_i w_i \ell_i
$$

avec :

- $\ell_i$ : loss de la transition $i$ ;
- $w_i$ : correction d'importance sampling ;
- $B$ : taille du batch.

Donc PER ne change pas seulement **quelles transitions on tire**. Il change aussi **combien chaque transition tiree pese dans la loss**.

Le role de $\beta$ est progressif :

- au debut de l'entrainement, on peut garder $\beta$ plus bas pour profiter fortement de la priorisation ;
- plus tard, on augmente $\beta$ vers 1 pour corriger davantage le biais statistique.

L'intuition finale :

- $\alpha$ controle **qui revient le plus souvent dans le batch** ;
- $\beta$ controle **a quel point on corrige le biais cree par ce choix** ;
- $p_i$ est mis a jour apres l'apprentissage, selon l'erreur ou la loss observee ;
- $w_i$ est utilise pendant le calcul de la loss, pour ponderer le gradient.

PER transforme donc le replay buffer en systeme de revision active : les transitions difficiles reviennent plus souvent, mais on evite qu'elles dominent completement l'apprentissage.
```

In [ ]:
class PrioritizedReplayBuffer:
    """Replay buffer priorise simple, pedagogique, sans sum-tree."""

    def __init__(self, capacity, alpha=0.6, beta=0.4, eps=1e-6):
        self.capacity = capacity
        self.alpha = alpha  # alpha dans P(i) : controle la force de la priorisation
        self.beta = beta    # beta dans w_i : controle la correction du biais
        self.eps = eps      # epsilon dans p_i = |delta_i| + eps
        self.storage = []
        self.priorities = np.zeros(capacity, dtype=np.float32)  # contient les p_i
        self.pos = 0
        self.max_priority = 1.0  # priorite donnee aux nouvelles transitions

    def push(self, state, action, reward, next_state, done):
        """Ajoute une transition (s, a, r, s', d) dans le buffer."""
        transition = (np.asarray(state, dtype=np.float32), int(action), float(reward),
                      np.asarray(next_state, dtype=np.float32), bool(done))

        if len(self.storage) < self.capacity:
            self.storage.append(transition)
        else:
            self.storage[self.pos] = transition

        # Une transition nouvelle n'a pas encore ete apprise :
        # on lui donne la priorite max pour qu'elle puisse etre tiree rapidement.
        self.priorities[self.pos] = self.max_priority

        # Buffer circulaire : quand il est plein, on remplace les plus anciennes transitions.
        self.pos = (self.pos + 1) % self.capacity

    def sample_probabilities(self):
        """Calcule P(i) = p_i^alpha / sum_j p_j^alpha."""
        active = self.priorities[:len(self.storage)]

        # p_i^alpha : alpha=0 donne uniforme, alpha=1 suit fortement les priorites.
        scaled = active ** self.alpha

        # Normalisation : les probabilites doivent sommer a 1.
        return scaled / scaled.sum()

    def sample(self, batch_size):
        """Tire un batch selon P(i) et renvoie aussi les poids d'importance w_i."""
        probs = self.sample_probabilities()

        # PER agit ici : on ne tire plus uniformement, mais selon P(i).
        indices = np.random.choice(len(self.storage), size=batch_size, replace=False, p=probs)

        # w_i = (1 / (N * P(i)))^beta
        # Si une transition est tiree trop souvent, son poids devient plus petit.
        weights = (len(self.storage) * probs[indices]) ** (-self.beta)

        # Normalisation des poids pour stabiliser les gradients.
        weights = weights / weights.max()

        batch = [self.storage[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)

        # indices sert plus tard a mettre a jour les priorites p_i du batch.
        # weights sert dans la loss : L = mean_i w_i * loss_i.
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones), indices, weights)

    def update_priorities(self, indices, td_errors):
        """Met a jour p_i = |delta_i| + eps apres une update."""
        # Les transitions encore mal predites gardent une grande priorite.
        new_priorities = np.abs(td_errors) + self.eps

        # On remplace uniquement les priorites des transitions qui etaient dans le batch.
        self.priorities[indices] = new_priorities

        # Les prochaines nouvelles transitions recevront la plus grande priorite connue.
        self.max_priority = max(self.max_priority, float(new_priorities.max()))

    def __len__(self):
        return len(self.storage)


per_buffer = PrioritizedReplayBuffer(capacity=10, alpha=0.6, beta=0.4)

# On remplit le buffer avec des transitions jouets.
for i in range(8):
    per_buffer.push(np.zeros(4), i % 2, 1.0, np.ones(4), False)

# On simule des erreurs TD : les dernieres transitions sont plus "surprenantes".
# Elles auront donc une plus grande priorite p_i.
per_buffer.update_priorities(
    np.arange(8),
    np.array([0.1, 0.2, 0.5, 0.7, 1.5, 2.0, 4.0, 8.0])
)

# Le sample renvoie le batch, les indices tires, et les poids d'importance w_i.
sample = per_buffer.sample(batch_size=4)

print("Buffer length:", len(per_buffer))
print("Sample indices:", sample[5])
print("IS weights    :", np.round(sample[6], 3))

In [ ]:
priorities = np.array([0.1, 0.2, 0.5, 0.7, 1.5, 2.0, 4.0, 8.0], dtype=np.float32)
alphas = [0.0, 0.3, 0.6, 1.0]
betas = [0.0, 0.4, 0.7, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for alpha in alphas:
    scaled = priorities ** alpha
    probs = scaled / scaled.sum()
    axes[0].plot(np.arange(len(priorities)), probs, marker="o", label=f"alpha={alpha}")
axes[0].set_title("PER: alpha transforme les priorites en probabilites")
axes[0].set_xlabel("Index transition (priorite croissante)")
axes[0].set_ylabel("P(i)")
axes[0].legend()

alpha = 0.6
probs = priorities ** alpha
probs = probs / probs.sum()
for beta in betas:
    weights = (len(priorities) * probs) ** (-beta)
    weights = weights / weights.max()
    axes[1].plot(np.arange(len(priorities)), weights, marker="o", label=f"beta={beta}")
axes[1].set_title("PER: beta corrige le biais d'echantillonnage")
axes[1].set_xlabel("Index transition (priorite croissante)")
axes[1].set_ylabel("Poids d'importance normalise")
axes[1].legend()

plt.tight_layout()
plt.show()


### Interpretation des courbes PER

Dans le panneau gauche, $\alpha=0$ est plat : toutes les transitions ont la meme probabilite. Quand $\alpha$ augmente, les transitions de droite, plus prioritaires, sont tirees beaucoup plus souvent. C'est le cote efficace de PER : le reseau revisite ses erreurs.

Dans le panneau droit, les poids d'importance font le mouvement inverse. Les transitions tres probables recoivent un poids plus faible, car elles sont deja surrepresentees dans le batch. Plus $\beta$ augmente, plus cette correction devient forte.

Il faut lire PER comme un compromis : apprendre plus vite en regardant les transitions surprenantes, mais corriger progressivement le biais que ce choix introduit. La prochaine brique agit sur un autre probleme : une cible 1-step propage parfois les rewards trop lentement.



## 5. n-step returns : faire voyager la recompense plus vite dans le passe

Dans DQN classique, la cible Bellman est une cible **1-step** :

$$
y_t = r_t + \gamma \max_{a'} Q_{\theta^-}(s_{t+1}, a')
$$

Elle regarde une seule recompense reelle, $r_t$, puis elle bootstrappe immediatement avec l'estimation du reseau cible pour l'etat suivant. C'est stable, parce qu'on ne depend pas trop d'une longue trajectoire observee. Mais le signal apprend lentement.

Imagine une trajectoire ou l'agent fait trois bonnes actions avant de recevoir une consequence importante. Avec une cible 1-step, le premier etat ne voit pas directement cette consequence. Elle doit remonter progressivement : d'abord vers l'etat juste avant, puis vers l'etat precedent, puis encore precedent. La recompense voyage donc dans le passe update apres update.

Les n-step returns changent cette cible. Au lieu de regarder seulement une recompense avant de bootstrapper, on accumule plusieurs rewards reels :

$$
G_t^{(n)} =
r_t + \gamma r_{t+1} + \dots + \gamma^{n-1}r_{t+n-1}
+ \gamma^n V(s_{t+n})
$$

Lecture terme par terme :

- $G_t^{(n)}$ est la cible n-step pour la transition qui commence en $s_t$ ;
- $r_t, r_{t+1}, \dots, r_{t+n-1}$ sont les vraies recompenses observees pendant les $n$ prochains pas ;
- chaque recompense future est reduite par une puissance de $\gamma$ ;
- $\gamma^n V(s_{t+n})$ est le bootstrap apres $n$ pas ;
- $V(s_{t+n})$ est souvent obtenu par un max sur les actions, ou dans Rainbow par l'esperance de la distribution de valeur.

Pour $n=1$, on retrouve exactement DQN classique :

$$
G_t^{(1)} = r_t + \gamma V(s_{t+1})
$$

Por $n=\infty$, on obtient le return monte-carlo complet, sans bootstrap :
$$
G_t^{(\infty)} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots
$$  

Pour $n=3$, la cible devient :

$$
G_t^{(3)} =
r_t + \gamma r_{t+1} + \gamma^2 r_{t+2}
+ \gamma^3 V(s_{t+3})
$$

La difference importante est donc celle-ci : avec $n=3$, l'etat $s_t$ recoit tout de suite l'information des trois prochaines recompenses observees. La recompense voyage plus vite vers le passe.

Sur CartPole, chaque pas donne souvent une recompense de survie. Si une sequence d'actions garde la tige droite pendant plusieurs pas, le n-step return permet d'associer plus vite les premiers etats de cette sequence a la survie qui suit. L'agent n'attend pas que cette information remonte uniquement par petits sauts de 1 pas.

Mais il y a un compromis.

- $n=1$ donne une cible courte, souvent moins bruitee, mais la propagation du signal est lente ;
- $n>1$ donne une cible plus informative, mais elle depend davantage de la trajectoire reellement observee ;
- si $n$ est trop grand, la cible peut devenir plus variable, surtout quand la politique explore beaucoup.

Rainbow utilise souvent $n=3$ parce que c'est un compromis pratique : assez long pour accelerer la propagation des rewards, assez court pour ne pas rendre la cible trop instable.

Dans la boucle de training, le n-step return s'integre avant le replay buffer. On ne stocke pas immediatement chaque transition 1-step. On garde un petit accumulateur temporaire :

```text
1. observer (s_t, a_t, r_t, s_{t+1}, done)
2. ajouter cette transition dans un buffer temporaire n-step
3. quand on a n transitions :
   - calculer r_t + gamma r_{t+1} + ... + gamma^{n-1} r_{t+n-1}
   - prendre comme next_state le s_{t+n}
   - stocker cette transition n-step dans le replay buffer
```

Donc le replay buffer ne contient plus exactement :

$$
(s_t, a_t, r_t, s_{t+1}, done)
$$

mais plutot :

$$
(s_t, a_t, R_t^{(n)}, s_{t+n}, done_n)
$$

avec :

- $R_t^{(n)}$ : somme des rewards observes sur $n$ pas ;
- $s_{t+n}$ : etat atteint apres ces $n$ pas ;
- $done_n$ : vrai si l'episode s'est termine pendant ces $n$ pas.

Si l'episode se termine avant les $n$ pas, on ne bootstrappe pas au-dela de l'etat terminal. La cible devient simplement la somme des rewards observes jusqu'a la fin.

Intuition finale : le n-step return est une facon de donner au reseau une cible un peu moins myope. DQN 1-step apprend en regardant juste le prochain pas ; Rainbow regarde quelques pas reels avant de demander au reseau cible de completer l'histoire.


In [ ]:
class NStepAccumulator:
    """Accumule des transitions et emet des transitions n-step."""

    def __init__(self, n_steps, gamma):
        self.n_steps = n_steps
        self.gamma = gamma
        self.buffer = deque()

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        emitted = []
        if done:
            while self.buffer:
                emitted.append(self._build(len(self.buffer)))
                self.buffer.popleft()
            return emitted
        if len(self.buffer) >= self.n_steps:
            emitted.append(self._build(self.n_steps))
            self.buffer.popleft()
        return emitted

    def _build(self, length):
        state, action, *_ = self.buffer[0]
        reward = sum((self.gamma ** k) * self.buffer[k][2] for k in range(length))
        next_state = self.buffer[length - 1][3]
        done = self.buffer[length - 1][4]
        return state, action, reward, next_state, done

    def flush(self):
        emitted = []
        while self.buffer:
            emitted.append(self._build(len(self.buffer)))
            self.buffer.popleft()
        return emitted


acc = NStepAccumulator(n_steps=3, gamma=0.9)
print(acc.push("s0", 0, 1.0, "s1", False))
print(acc.push("s1", 1, 1.0, "s2", False))
print(acc.push("s2", 0, 1.0, "s3", False))
print("Reward attendu:", 1 + 0.9 + 0.9**2)


In [ ]:
gamma = 0.95
rewards = np.array([0, 0, 0, 0, 1, 1, 1], dtype=np.float32)  # reward tardive
bootstrap_value = 5.0
n_options = [1, 3, 5]

def n_step_target_at(t, n):
    total = 0.0
    for k in range(n):
        if t + k < len(rewards):
            total += (gamma ** k) * rewards[t + k]
    return total + (gamma ** n) * bootstrap_value

targets = {n: [n_step_target_at(t, n) for t in range(len(rewards))] for n in n_options}

plt.figure(figsize=(10, 4))
for n, vals in targets.items():
    plt.plot(vals, marker="o", linewidth=2, label=f"n={n}")
plt.bar(np.arange(len(rewards)), rewards, alpha=0.2, color="gray", label="rewards observees")
plt.title("Une reward tardive influence plus tot les cibles n-step")
plt.xlabel("Position t dans la trajectoire")
plt.ylabel("Cible utilisee pour l'update")
plt.legend()
plt.tight_layout()
plt.show()


### Interpretation du plot n-step

Les barres grises indiquent que les rewards positives arrivent tard. Avec $n=1$, les premiers etats ne voient presque pas ces rewards directement : ils dependent surtout du bootstrap. Avec $n=3$ ou $n=5$, la cible des etats plus anciens incorpore plus vite les rewards observees.

Ce qu'il faut regarder : les courbes n-step montent plus tot sur l'axe des positions $t$. C'est exactement l'effet recherche : propager le signal utile vers les decisions qui l'ont rendu possible.

Le prix a payer est que la cible depend de plusieurs transitions successives. Elle peut donc etre plus variable. Rainbow compense en combinant n-step avec replay priorise et target network. La prochaine brique change encore la nature de ce que le reseau predit : pas seulement une moyenne, mais une distribution.


## 6. Distributional RL / C51 : apprendre une distribution, pas seulement une moyenne

Jusqu'ici, DQN apprend une seule valeur par action :

$$
Q(s,a) = \mathbb{E}[G_t \mid s_t=s, a_t=a]
$$

Cette valeur est une **moyenne** du retour futur. C'est utile pour choisir une action, mais une moyenne peut cacher des situations tres differentes.

Prenons un exemple volontairement simple. Depuis le meme etat, deux actions ont le meme retour moyen :

```text
Action A : 5, 5, 5, 5, 5        moyenne = 5
Action B : 0, 0, 5, 10, 10      moyenne = 5
```

Pour DQN, ces deux actions peuvent sembler equivalentes, parce que leur esperance est la meme. Pourtant elles ne racontent pas la meme histoire :

- l'action A est reguliere : on sait presque toujours ce qui va se passer ;
- l'action B est plus incertaine : parfois elle echoue, parfois elle rapporte beaucoup ;
- selon l'etat, l'exploration, ou la suite de l'apprentissage, cette information peut etre importante.

Distributional RL part de cette idee : au lieu d'apprendre seulement **la moyenne** du retour, on apprend une approximation de **la distribution complete** des retours possibles.

Autrement dit, on ne demande plus au reseau :

> "combien vaut cette action en moyenne ?"

On lui demande plutot :

> "quelles valeurs de retour sont possibles, et avec quelles probabilites ?"

C51 est une version simple et concrete de cette idee. Le nom vient de "Categorical 51", parce que l'article original utilise 51 points fixes, appeles **atomes**, pour representer la distribution.

### Le support fixe

C51 commence par choisir une grille de valeurs possibles :

$$
z_i = v_{\min} + i\Delta z,
\qquad i=0,\dots,N-1
$$

avec :

- $z_i$ : le $i$-eme atome de valeur ;
- $v_{\min}$ : la valeur minimale representable ;
- $v_{\max}$ : la valeur maximale representable ;
- $N$ : le nombre d'atomes ;
- $\Delta z = \frac{v_{\max}-v_{\min}}{N-1}$ : l'ecart entre deux atomes.

Par exemple, avec $v_{\min}=0$, $v_{\max}=10$ et $N=6$, le support est :

```text
z = [0, 2, 4, 6, 8, 10]
```

Le reseau ne sort donc plus seulement :

```text
Q(s, gauche), Q(s, droite)
```

Il sort une distribution de probabilites pour chaque action :

```text
action gauche : [0.00, 0.05, 0.20, 0.50, 0.20, 0.05]
action droite : [0.20, 0.10, 0.10, 0.10, 0.10, 0.40]
```

Chaque ligne somme a 1. Chaque probabilite dit : "quelle masse de probabilite je mets sur cet atome de retour".
Ici, il faut vraiment voir les **atomes** comme des petites cases fixes sur une regle graduee.

C51 ne laisse pas le reseau predire n'importe quelle valeur continue comme $5.37$ ou $8.91$. Il lui donne une grille fixe de valeurs possibles :

```text
0      2      4      6      8      10
|------|------|------|------|------|
z0     z1     z2     z3     z4     z5
```

Chaque position de cette grille est un **atome**. Un atome n'est donc pas une transition, ni une action, ni un neurone special au sens conceptuel. C'est simplement une valeur possible du retour futur.

Par exemple, l'atome $z_3 = 6$ veut dire :

> "une partie de ma croyance dit que le retour futur pourrait etre autour de 6."

La **masse** mise sur un atome est la probabilite que le reseau attribue a cette valeur de retour. Si on ecrit :

```text
action gauche : [0.00, 0.05, 0.20, 0.50, 0.20, 0.05]
support       : [0,    2,    4,    6,    8,    10]
```

cela se lit comme :

- probabilite 0.00 que le retour soit autour de 0 ;
- probabilite 0.05 que le retour soit autour de 2 ;
- probabilite 0.20 que le retour soit autour de 4 ;
- probabilite 0.50 que le retour soit autour de 6 ;
- probabilite 0.20 que le retour soit autour de 8 ;
- probabilite 0.05 que le retour soit autour de 10.

Donc la distribution dit :

> "pour l'action gauche, je pense surtout que le retour sera proche de 6, avec un peu de chance d'etre proche de 4 ou 8."

On peut l'imaginer comme des piles de sable posees sur une regle :

```text
retour futur :   0      2      4      6      8      10
masse gauche :  0.00   0.05   0.20   0.50   0.20   0.05
                                 ▂      █      ▂
```

Plus la masse est grande sur un atome, plus le reseau croit que le retour futur tombera pres de cette valeur.

La somme des masses vaut toujours 1, comme pour toute distribution de probabilite :

```text
0.00 + 0.05 + 0.20 + 0.50 + 0.20 + 0.05 = 1.00
```

C'est pour cela qu'on parle de distribution : le reseau repartit une quantite totale de croyance egale a 1 entre plusieurs valeurs possibles.

La difference avec DQN devient alors tres concrete :

```text
DQN :
  action gauche -> Q = 6.0

C51 :
  action gauche -> 0 avec proba 0.00
                   2 avec proba 0.05
                   4 avec proba 0.20
                   6 avec proba 0.50
                   8 avec proba 0.20
                  10 avec proba 0.05
```

DQN dit seulement : "en moyenne, je pense que ca vaut 6".

C51 dit : "je pense que ca vaut en moyenne 6, mais voici aussi comment mon incertitude est repartie autour de cette moyenne. Je pense que dans $50\%$ des cas le retour/gain vaudra 6, que dans $20\%$ des cas il vaudra 4, que dans $20\%$ des cas il vaudra 8, et que dans $5\%$ des cas il vaudra 2 ou 10."

Important : les atomes ne bougent pas pendant l'apprentissage. Ce sont des graduations fixes. Ce que le reseau apprend, ce sont les masses/probabilites a mettre sur ces graduations pour chaque couple `(etat, action)`.


### Recuperer Q(s,a)

Meme si le reseau apprend une distribution, on a encore besoin d'une valeur scalaire pour choisir l'action. On recupere donc l'esperance de la distribution :

$$
Q(s,a)=\sum_i z_i p_i(s,a)
$$

Lecture terme par terme :

- $z_i$ est la valeur de l'atome ;
- $p_i(s,a)$ est la probabilite predite sur cet atome pour l'action $a$ ;
- $z_i p_i(s,a)$ est la contribution de cet atome a la moyenne ;
- la somme donne le retour moyen, donc l'equivalent de $Q(s,a)$.

C'est important : C51 ne remplace pas l'idee de choisir l'action avec une valeur. Il enrichit ce que le reseau apprend. Pour agir, Rainbow prend souvent :

$$
argmax_a E[Z(s,a)]
$$

c'est-a-dire l'action dont la distribution a la meilleure esperance.

### Exemple de retours de même moyenne

Supposons ce support :

```text
z = [0, 2, 4, 6, 8, 10]
```

Et deux distributions :

```text
Action A : [0.00, 0.00, 0.20, 0.60, 0.20, 0.00]
Action B : [0.40, 0.00, 0.00, 0.00, 0.00, 0.60]
```

L'action A concentre sa masse autour de l'atome 6. Elle est assez reguliere.

L'action B met beaucoup de masse sur l'atome 0 et l'atome 10. Elle est plus bimodale : elle peut etre tres mauvaise ou tres bonne.

Le calcul de l'esperance donne :

```text
E[A] = 0*0.00 + 2*0.00 + 4*0.20 + 6*0.60 + 8*0.20 + 10*0.00 = 6.0
E[B] = 0*0.40 + 2*0.00 + 4*0.00 + 6*0.00 + 8*0.00 + 10*0.60 = 6.0
```

Les deux actions ont la meme moyenne, mais leurs distributions sont tres differentes. C'est exactement l'information que DQN scalaire perd.

Sur CartPole, les rewards sont simples, mais l'idee reste utile pedagogiquement : l'agent peut representer non seulement "combien de temps je pense survivre en moyenne", mais aussi "quelle gamme de retours je crois possible depuis cet etat".

### La cible distributionnelle

Dans DQN, la cible scalaire ressemble a :

$$
y = r + \gamma * Q_{target}(s', a*)
$$

Dans C51, la cible est une **distribution de valeurs**. On prend la distribution predite par le reseau cible au prochain etat, puis on la decale avec la recompense :
valeur cible de chaque $\text{atome} = r + \gamma \times z_i$


Si on utilise du n-step, cela devient :

```text
valeur cible de chaque atome = R_t^(n) + gamma^n * z_i
```

Donc si le reseau cible pense que le prochain retour peut etre sur les atomes $[0, 2, 4, 6]$, apres une recompense immediate de $1$, ces valeurs deviennent environ :

$$
[1 + \gamma*0, 1 + \gamma*2, 1 + \gamma*4, 1 + \gamma*6]
$$

La distribution est donc transportee vers la gauche ou la droite selon la recompense observee.

### Pourquoi il faut projeter

Le probleme est que ces nouvelles valeurs ne tombent pas forcement exactement sur les atomes fixes du support.

Exemple :

$$
\text{support fixe : } [0, 2, 4, 6, 8, 10]
\text{valeur cible : } 5.3
$$

Il n'y a pas d'atome $5.3$. C51 doit donc repartir cette masse entre les deux atomes voisins :

```text
5.3 est entre 4 et 6
plus proche de 6 que de 4
donc on met une partie de la masse sur 4 et une partie sur 6
```

C'est le role de la projection :

$$
m = \Pi_{\mathcal{Z}} T Z
$$

Lecture intuitive :

- $Z$ est la distribution predite par le reseau cible ;
- $T Z$ est cette distribution apres application de la cible Bellman distributionnelle ;
- $\Pi_{\mathcal{Z}}$ reprojette cette distribution sur le support fixe ;
- $m$ est la distribution cible finale utilisee dans la loss.

Donc la projection ne change pas le principe de Bellman. Elle resout seulement un probleme pratique : le reseau predit sur une grille fixe, mais la cible Bellman produit des valeurs continues.

### Exemple de projection

Imaginons qu'une masse de probabilite $0.7$ arrive sur la valeur cible $5.3$.

Le support contient $4$ et $6$. Comme $5.3$ est entre les deux :

```text
distance a 4 : 1.3
distance a 6 : 0.7
```

On donne plus de masse a $6$, parce que $5.3$ en est plus proche :

```text
masse vers 4 :  0.7 * (6 - 5.3) / (6 - 4) = 0.245
masse vers 6 :  0.7 * (5.3 - 4) / (6 - 4) = 0.455
```

La masse totale est conservee :

```text
0.245 + 0.455 = 0.700
```

C'est exactement ce que fait C51 pour chaque atome de la distribution cible.

### La loss C51

Puisque le reseau predit une distribution de probabilites, on n'utilise plus une MSE directe sur un scalaire. On compare deux distributions :

- la distribution cible projetee $m$ ;
- la distribution predite par le reseau courant $p_\theta(s,a)$.

La loss est une cross-entropy :

$$
\mathcal{L}
=
-\sum_i m_i \log p_{\theta,i}(s,a)
$$

Intuition :

- si le reseau met beaucoup de probabilite la ou la cible met beaucoup de masse, la loss est faible ;
- si le reseau met peu de probabilite sur les atomes importants de la cible, la loss augmente ;
- le reseau apprend donc a deplacer sa distribution vers la distribution cible.

Avec PER, cette loss par transition peut ensuite servir a mettre a jour la priorite : une transition dont la distribution est mal predite reste prioritaire.

### Ce qu'il faut retenir

C51 remplace la prediction scalaire :

$$
Q(s,a)
$$

par une prediction distributionnelle :

$$
P(Z(s,a)=z_i) \quad \text{pour chaque atome } z_i
$$

Mais on garde l'esperance pour choisir les actions :

$$
Q(s,a) = E[Z(s,a)]
$$

Le benefice est que le reseau recoit un signal d'apprentissage plus riche. Il n'apprend pas seulement "la bonne moyenne", il apprend aussi comment la masse de retour se repartit entre plusieurs valeurs possibles.

La difficulte supplementaire vient de la projection : apres Bellman, les valeurs cibles ne tombent pas exactement sur la grille fixe, donc C51 redistribue la masse sur les atomes voisins.


In [ ]:
class CategoricalDuelingNetwork(nn.Module):
    """Reseau dueling qui predit une distribution C51 par action."""

    def __init__(self, obs_dim, n_actions, hidden_dim=64, n_atoms=21, v_min=-10.0, v_max=10.0):
        super().__init__()
        self.n_actions = n_actions
        self.n_atoms = n_atoms
        self.register_buffer("support", torch.linspace(v_min, v_max, n_atoms))
        self.features = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.value = nn.Linear(hidden_dim, n_atoms)
        self.advantage = nn.Linear(hidden_dim, n_actions * n_atoms)

    def dist(self, obs):
        h = self.features(obs)
        value = self.value(h).view(-1, 1, self.n_atoms)
        advantage = self.advantage(h).view(-1, self.n_actions, self.n_atoms)
        logits = value + advantage - advantage.mean(dim=1, keepdim=True)
        return F.softmax(logits, dim=-1)

    def q_values(self, obs):
        probs = self.dist(obs)
        return torch.sum(probs * self.support.view(1, 1, -1), dim=-1)


cat_net = CategoricalDuelingNetwork(4, 2, n_atoms=21)
obs_batch = torch.randn(6, 4)
probs = cat_net.dist(obs_batch)
q_values = cat_net.q_values(obs_batch)
print("Distribution shape:", tuple(probs.shape))
print("Q-values shape    :", tuple(q_values.shape))
print("Mass check action0:", probs[0, 0].sum().item())


In [ ]:
support = np.linspace(0, 10, 51)
# Deux distributions de meme moyenne 5, mais formes differentes.
stable = np.exp(-0.5 * ((support - 5.0) / 0.7) ** 2)
stable = stable / stable.sum()
risky = 0.5 * np.exp(-0.5 * ((support - 1.5) / 0.5) ** 2) + 0.5 * np.exp(-0.5 * ((support - 8.5) / 0.5) ** 2)
risky = risky / risky.sum()
mean_stable = (support * stable).sum()
mean_risky = (support * risky).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(support, stable, linewidth=2, label=f"stable, moyenne={mean_stable:.2f}")
axes[0].plot(support, risky, linewidth=2, label=f"risquee, moyenne={mean_risky:.2f}")
axes[0].set_title("Deux distributions presque meme Q, risques differents")
axes[0].set_xlabel("Retour possible")
axes[0].set_ylabel("Probabilite")
axes[0].legend()

atoms = np.linspace(-2, 8, 11)
target_value = 3.7
lower = np.searchsorted(atoms, target_value) - 1
upper = lower + 1
lower = np.clip(lower, 0, len(atoms) - 1)
upper = np.clip(upper, 0, len(atoms) - 1)
projection = np.zeros_like(atoms)
if lower == upper:
    projection[lower] = 1.0
else:
    projection[lower] = (atoms[upper] - target_value) / (atoms[upper] - atoms[lower])
    projection[upper] = (target_value - atoms[lower]) / (atoms[upper] - atoms[lower])
axes[1].bar(atoms, projection, width=0.7, color="darkorange")
axes[1].axvline(target_value, color="black", linestyle="--", label=f"cible {target_value}")
axes[1].set_title("Projection C51 sur les atomes voisins")
axes[1].set_xlabel("Atomes fixes z_i")
axes[1].set_ylabel("Masse projetee")
axes[1].legend()

plt.tight_layout()
plt.show()


### Interpretation des distributions C51

Dans le panneau gauche, les deux actions ont presque la meme esperance. Un DQN scalaire les verrait donc comme presque equivalentes. C51 conserve plus d'information : une action est concentree autour de sa moyenne, l'autre est bimodale et donc plus incertaine.

Le panneau droit montre la projection. La cible `3.7` n'est pas exactement un atome. On distribue donc sa masse entre les deux atomes voisins. Pour une distribution complete, on fait cette operation pour toute la masse de probabilite.

La consequence importante : la loss Rainbow n'est plus une MSE entre deux scalaires. On compare une distribution cible projetee et la distribution predite par le reseau, avec une cross-entropy. La derniere brique change l'exploration : au lieu d'ajouter du hasard aux actions, on ajoute du bruit aux parametres.


## 7. Noisy Nets : faire venir l'exploration des parametres

Dans DQN classique, l'exploration vient souvent de l'epsilon-greedy. A chaque pas, on fait :

$$
a_t =
\begin{cases}
\arg\max\limits_a Q_\theta(s_t,a), & \text{avec probabilite } 1-\varepsilon, \\
\text{action aleatoire dans } \mathcal{A}, & \text{avec probabilite } \varepsilon.
\end{cases}
$$
C'est simple et efficace, mais c'est aussi assez brutal. Quand l'agent explore, il ignore completement ce que le reseau a appris. Il ne teste pas une hypothese coherente, il lance juste une action au hasard.

Sur CartPole, cela peut vouloir dire :

> "je pense qu'il faut pousser a droite, mais epsilon vient de tomber, donc je pousse a gauche sans raison particuliere."

Ce hasard peut aider au debut, mais il devient moins elegant quand on veut une exploration plus liee a l'incertitude du modele.

Noisy Nets proposent une autre idee : au lieu d'ajouter du hasard **dans le choix de l'action**, on ajoute du hasard **dans les parametres du reseau**.

Une couche lineaire classique calcule :

$$
y = Wx + b
$$

Une couche NoisyLinear calcule plutot :

$$
y = (W + \sigma_W \odot \epsilon_W)x + (b + \sigma_b \odot \epsilon_b)
$$

Lecture terme par terme :

- $W$ et $b$ sont les poids et biais moyens, comme dans une couche lineaire normale ;
- $\epsilon_W$ et $\epsilon_b$ sont des bruits aleatoires resamples pendant l'entrainement ;
- $\sigma_W$ et $\sigma_b$ controlent l'amplitude du bruit ;
- $\odot$ signifie multiplication element par element ;
- $(W + \sigma_W \odot \epsilon_W)$ est donc une version legerement perturbee des poids.

La difference est subtile mais importante.

Avec epsilon-greedy, la politique apprise reste la meme, mais on la remplace parfois par du hasard pur.

Avec NoisyNet, la politique elle-meme change legerement a chaque resampling du bruit. L'agent teste donc une variante coherente de son reseau :

```text
epsilon-greedy :
  "je connais ma politique, mais parfois je fais n'importe quoi"

NoisyNet :
  "je teste une version legerement differente de ma politique"
```

C'est une exploration plus structuree. Si le bruit change les parametres qui influencent plusieurs actions dans plusieurs etats proches, l'agent peut explorer une strategie un peu differente pendant un moment, au lieu de faire seulement des actions isolees au hasard.

### Pourquoi sigma est apprenable ?

Dans NoisyNet, $\sigma$ n'est pas seulement un hyperparametre fixe. Il est appris par gradient comme les autres parametres du reseau.

Cela signifie que le reseau peut apprendre ou il a besoin de bruit :

- si une partie du reseau est utile et stable, $\sigma$ peut devenir petit ;
- si une partie du reseau reste incertaine, $\sigma$ peut rester plus grand ;
- l'exploration devient donc adaptee a l'apprentissage.

C'est une difference importante avec epsilon-greedy : epsilon suit souvent un planning manuel, par exemple on le fait decroitre de $1.0$ a $0.05$. Le reseau ne decide pas directement ou explorer. Avec NoisyNet, une partie de cette decision est integree dans les parametres.

### Quand resampler le bruit ?

Pendant l'entrainement, on resample le bruit avant de choisir une action, et souvent aussi avant certaines updates. Cela donne au reseau une nouvelle version perturbee de ses parametres.

Dans Rainbow, l'action est choisie comme :

```text
resampler le bruit NoisyLinear
calculer les valeurs des actions
prendre argmax_a E[Z(s,a)]
```

Il n'y a donc pas besoin d'epsilon-greedy comme mecanisme principal. L'exploration vient du fait que les valeurs estimees changent legerement avec le bruit.

En evaluation, au contraire, on veut mesurer la politique moyenne apprise. On coupe donc le bruit, ou on utilise les parametres moyens :

```text
evaluation : y = Wx + b
```

Cela rend l'evaluation deterministe et comparable.

### Exemple : le bruit paramétrique peut changer l'action

Supposons que pour un etat donne, le reseau moyen estime :

```text
Q(gauche) = 5.00
Q(droite) = 5.10
```

Sans exploration, l'agent choisit toujours `droite`.

Avec epsilon-greedy, il choisira parfois `gauche`, mais uniquement parce qu'une action aleatoire a ete tiree.

Avec NoisyNet, une version perturbee du reseau peut produire :

```text
resampling 1 : Q(gauche) = 5.25, Q(droite) = 5.05 -> gauche
resampling 2 : Q(gauche) = 4.90, Q(droite) = 5.30 -> droite
resampling 3 : Q(gauche) = 5.15, Q(droite) = 5.12 -> gauche
```

L'agent explore parce que son estimation change legerement. Ce n'est pas une action aleatoire deconnectee du reseau : c'est une action choisie par une version alternative du reseau.

### Ce que NoisyNet corrige dans Rainbow

NoisyNet remplace donc la partie la plus artificielle de DQN : le hasard externe.

Dans Rainbow :

- Double DQN corrige la surestimation ;
- Dueling organise mieux la prediction de valeur ;
- PER choisit mieux les transitions a reviser ;
- n-step propage plus vite les rewards ;
- C51 apprend une distribution de retours ;
- NoisyNet fournit une exploration apprise et structuree.

Intuition finale : epsilon-greedy explore au niveau des actions, NoisyNet explore au niveau des hypotheses du reseau. C'est pour cela qu'il s'integre bien a Rainbow : l'agent ne fait pas juste "n'importe quoi" de temps en temps, il essaye des variantes coherentes de ce qu'il croit deja savoir.


In [ ]:
class NoisyLinear(nn.Module):
    """Couche lineaire bruit ee factorisee, version pedagogique."""

    def __init__(self, in_features, out_features, std_init=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))
        self.register_buffer("weight_epsilon", torch.empty(out_features, in_features))
        self.register_buffer("bias_epsilon", torch.empty(out_features))
        self.std_init = std_init
        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        bound = 1 / np.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-bound, bound)
        self.bias_mu.data.uniform_(-bound, bound)
        self.weight_sigma.data.fill_(self.std_init / np.sqrt(self.in_features))
        self.bias_sigma.data.fill_(self.std_init / np.sqrt(self.out_features))

    def _scale_noise(self, size):
        x = torch.randn(size)
        return x.sign() * x.abs().sqrt()

    def reset_noise(self):
        eps_in = self._scale_noise(self.in_features)
        eps_out = self._scale_noise(self.out_features)
        self.weight_epsilon.copy_(torch.outer(eps_out, eps_in))
        self.bias_epsilon.copy_(eps_out)

    def forward(self, x):
        if self.training:
            weight = self.weight_mu + self.weight_sigma * self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)


noisy = NoisyLinear(4, 2)
probe = torch.tensor([[0.1, 0.0, 0.05, 0.2]], dtype=torch.float32)
noisy.train()
out1 = noisy(probe)
noisy.reset_noise()
out2 = noisy(probe)
noisy.eval()
out_eval1 = noisy(probe)
out_eval2 = noisy(probe)
print("Train outputs differ after reset:", not torch.allclose(out1, out2))
print("Eval outputs stable          :", torch.allclose(out_eval1, out_eval2))


In [ ]:
# Comparaison conceptuelle epsilon-greedy vs NoisyNet sur un etat fixe.
base_q = np.array([1.0, 0.85])  # le reseau prefere legerement gauche
n_samples = 1000
epsilon = 0.2

rng = np.random.default_rng(42)
eps_actions = []
for _ in range(n_samples):
    if rng.random() < epsilon:
        eps_actions.append(rng.integers(2))
    else:
        eps_actions.append(int(base_q.argmax()))
eps_actions = np.array(eps_actions)

noisy_actions = []
noisy_qs = []
for _ in range(n_samples):
    perturbed_q = base_q + rng.normal(0, 0.18, size=2)
    noisy_qs.append(perturbed_q)
    noisy_actions.append(int(perturbed_q.argmax()))
noisy_qs = np.array(noisy_qs)
noisy_actions = np.array(noisy_actions)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(["gauche", "droite"], [(eps_actions == 0).mean(), (eps_actions == 1).mean()], color=["steelblue", "tomato"])
axes[0].set_ylim(0, 1)
axes[0].set_title("epsilon-greedy: hasard externe")
axes[0].set_ylabel("Frequence de choix")

axes[1].hist(noisy_qs[:, 0] - noisy_qs[:, 1], bins=35, color="purple", alpha=0.75)
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("NoisyNet: preference Q_gauche - Q_droite")
axes[1].set_xlabel("Difference de preference apres bruit")
axes[1].set_ylabel("Frequence")

plt.tight_layout()
plt.show()
print("Frequence droite epsilon-greedy:", round((eps_actions == 1).mean(), 3))
print("Frequence droite NoisyNet      :", round((noisy_actions == 1).mean(), 3))


### Interpretation epsilon-greedy vs NoisyNet

Dans le panneau gauche, epsilon-greedy choisit parfois l'action non preferee uniquement parce qu'un interrupteur aleatoire s'est active. L'action aleatoire ne depend pas de la confiance du reseau.

Dans le panneau droit, on regarde la difference `Q_gauche - Q_droite` apres perturbation. Quand cette difference passe sous zero, la preference change on selecttionnera l'action droite. L'exploration vient donc d'une politique perturbee, pas d'une action tiree au hasard apres coup.

Rainbow utilise Noisy Nets parce que cette exploration est compatible avec un agent greedy : on choisit toujours l'argmax, mais l'argmax d'un reseau legerement bruite. Il nous reste maintenant a assembler toutes les pieces.


## 8. Rainbow : assembler les composants

On peut maintenant lire Rainbow comme une table de corrections.

| Brique | Probleme corrige | Ou elle agit |
|---|---|---|
| Double DQN | max trop optimiste | choix/evaluation de l'action suivante |
| Dueling | `Q(s,a)` melange valeur d'etat et preference d'action | architecture du reseau |
| PER | replay uniforme gaspille des updates | sampling du batch |
| n-step | reward propagee lentement | construction des transitions |
| C51 | une moyenne cache la distribution des retours | sortie du reseau et loss |
| Noisy Nets | epsilon-greedy explore brutalement | couches lineaires de l'actor-value |

La difficulte de Rainbow n'est pas une equation unique. C'est le fait que toutes ces corrections doivent se parler correctement. Par exemple, PER sample des transitions n-step; C51 transforme la cible en distribution; Double DQN choisit l'action avec l'esperance de cette distribution; NoisyNet modifie la politique de selection pendant l'entrainement.

Les sources primaires a garder en tete pendant la lecture sont : **Double DQN** ([van Hasselt et al., 2016](https://arxiv.org/abs/1509.06461)), **Dueling** ([Wang et al., 2016](https://arxiv.org/abs/1511.06581)), **PER** ([Schaul et al., 2016](https://arxiv.org/abs/1511.05952)), **C51** ([Bellemare et al., 2017](https://arxiv.org/abs/1707.06887)), **Noisy Nets** ([Fortunato et al., 2018](https://arxiv.org/abs/1706.10295)) et l'assemblage **Rainbow** ([Hessel et al., 2018](https://arxiv.org/abs/1710.02298)).

Le code suivant reste pedagogique : il privilegie la lisibilite sur la vitesse. Un vrai Rainbow Atari utiliserait notamment un sum-tree pour PER et des optimisations plus agressives.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    classDef result fill:none,stroke:#16a34a,stroke-width:2px
    classDef warning fill:none,stroke:#dc2626,stroke-width:2px

    S["État courant s_t"] --> N["RainbowNetwork online"]
    N --> Q["Espérances Q(s_t,a)"]
    Q --> A["Action greedy sous bruit NoisyNet"]
    A --> E["Interaction environnement"]
    E --> NS["Accumulateur n-step"]
    NS --> PER["Replay priorisé"]
    PER --> B["Batch pondéré"]
    B --> SEL["Sélection online de a*"]
    SEL --> TGT["Évaluation target de Z(s' ,a*)"]
    TGT --> PROJ["Projection C51 sur le support fixe"]
    PROJ --> LOSS["Cross-entropy distributionnelle pondérée"]
    LOSS --> UPD["Update online + priorités PER"]
    UPD --> SYNC["Copie périodique vers target"]

    class S,N,Q,A,E primary
    class NS,PER,B,SEL,TGT secondary
    class PROJ,LOSS,UPD,SYNC result
```

In [ ]:
class RainbowNetwork(nn.Module):
    """Dueling + Noisy + C51 pour actions discretes."""

    def __init__(self, obs_dim, n_actions, hidden_dim=64, n_atoms=21, v_min=-10.0, v_max=10.0):
        super().__init__()
        self.n_actions = n_actions
        self.n_atoms = n_atoms
        self.register_buffer("support", torch.linspace(v_min, v_max, n_atoms))
        self.features = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.value = nn.Sequential(NoisyLinear(hidden_dim, hidden_dim), nn.ReLU(), NoisyLinear(hidden_dim, n_atoms))
        self.advantage = nn.Sequential(NoisyLinear(hidden_dim, hidden_dim), nn.ReLU(), NoisyLinear(hidden_dim, n_actions * n_atoms))

    def reset_noise(self):
        for module in self.modules():
            if isinstance(module, NoisyLinear):
                module.reset_noise()

    def dist(self, obs):
        h = self.features(obs)
        value = self.value(h).view(-1, 1, self.n_atoms)
        advantage = self.advantage(h).view(-1, self.n_actions, self.n_atoms)
        logits = value + advantage - advantage.mean(dim=1, keepdim=True)
        return F.softmax(logits, dim=-1).clamp(min=1e-6)

    def q_values(self, obs):
        probs = self.dist(obs)
        return torch.sum(probs * self.support.view(1, 1, -1), dim=-1)


rainbow_net = RainbowNetwork(obs_dim=4, n_actions=2)
obs = torch.randn(3, 4)
print("dist shape:", tuple(rainbow_net.dist(obs).shape))
print("q shape   :", tuple(rainbow_net.q_values(obs).shape))
print("mass check:", rainbow_net.dist(obs)[0, 0].sum().item())


In [ ]:
def projection_distribution(next_dist, rewards, dones, gamma, n_step, support, v_min, v_max):
    """Projette la distribution cible C51 sur le support fixe."""
    batch_size, n_atoms = next_dist.shape
    delta_z = (v_max - v_min) / (n_atoms - 1)
    projected = torch.zeros_like(next_dist)

    tz = rewards[:, None] + (1.0 - dones[:, None]) * (gamma ** n_step) * support[None, :]
    tz = tz.clamp(v_min, v_max)
    b = (tz - v_min) / delta_z
    lower = b.floor().long()
    upper = b.ceil().long()

    for atom in range(n_atoms):
        l = lower[:, atom]
        u = upper[:, atom]
        mass = next_dist[:, atom]
        same = l == u
        batch_idx = torch.arange(batch_size)
        projected[batch_idx, l] += mass * same.float()
        projected[batch_idx, l] += mass * (u.float() - b[:, atom]) * (~same).float()
        projected[batch_idx, u] += mass * (b[:, atom] - l.float()) * (~same).float()

    return projected / projected.sum(dim=1, keepdim=True).clamp(min=1e-8)


support_t = torch.linspace(-10, 10, 21)
next_dist = torch.full((2, 21), 1 / 21)
rewards_t = torch.tensor([1.0, -1.0])
dones_t = torch.tensor([0.0, 1.0])
projected = projection_distribution(next_dist, rewards_t, dones_t, 0.99, 3, support_t, -10, 10)
print("Projected shape:", tuple(projected.shape))
print("Masses         :", projected.sum(dim=1).numpy().round(5))


In [ ]:
class RainbowDQNAgent:
    """Agent Rainbow compact pour illustrer le flux complet."""

    def __init__(self, obs_dim, n_actions, hidden_dim=64, n_atoms=21, v_min=0.0, v_max=500.0,
                 gamma=0.99, n_steps=3, batch_size=64, lr=1e-3, capacity=10_000):
        self.n_actions = n_actions
        self.gamma = gamma
        self.n_steps = n_steps
        self.batch_size = batch_size
        self.v_min = v_min
        self.v_max = v_max
        self.online = RainbowNetwork(obs_dim, n_actions, hidden_dim, n_atoms, v_min, v_max)
        self.target = copy.deepcopy(self.online)
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=lr)
        self.replay = PrioritizedReplayBuffer(capacity=capacity)
        self.n_step = NStepAccumulator(n_steps=n_steps, gamma=gamma)
        self.updates = 0

    def select_action(self, obs, deterministic=False):
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        self.online.eval() if deterministic else self.online.train()
        if not deterministic:
            self.online.reset_noise()
        with torch.no_grad():
            q = self.online.q_values(obs_t)
        return int(q.argmax(dim=1).item())

    def store_transition(self, obs, action, reward, next_obs, done):
        for transition in self.n_step.push(obs, action, reward, next_obs, done):
            self.replay.push(*transition)

    def learn_step(self):
        if len(self.replay) < self.batch_size:
            return {}
        states, actions, rewards, next_states, dones, indices, weights = self.replay.sample(self.batch_size)
        states = torch.as_tensor(states, dtype=torch.float32)
        actions = torch.as_tensor(actions, dtype=torch.long)
        rewards = torch.as_tensor(rewards, dtype=torch.float32)
        next_states = torch.as_tensor(next_states, dtype=torch.float32)
        dones = torch.as_tensor(dones, dtype=torch.float32)
        weights = torch.as_tensor(weights, dtype=torch.float32)

        self.online.train()
        self.target.train()
        self.online.reset_noise()
        self.target.reset_noise()

        dist = self.online.dist(states)
        chosen_dist = dist[torch.arange(states.shape[0]), actions]
        q_pred = torch.sum(chosen_dist * self.online.support[None, :], dim=1)

        with torch.no_grad():
            next_actions = self.online.q_values(next_states).argmax(dim=1)
            next_dist_all = self.target.dist(next_states)
            next_dist = next_dist_all[torch.arange(states.shape[0]), next_actions]
            target_dist = projection_distribution(next_dist, rewards, dones, self.gamma, self.n_steps,
                                                  self.online.support, self.v_min, self.v_max)
            target_q = torch.sum(next_dist * self.online.support[None, :], dim=1)
            td_error = (rewards + (self.gamma ** self.n_steps) * (1 - dones) * target_q - q_pred).abs()

        per_sample_loss = -(target_dist * torch.log(chosen_dist.clamp(min=1e-6))).sum(dim=1)
        loss = (weights * per_sample_loss).mean()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.replay.update_priorities(indices, td_error.detach().numpy() + 1e-6)
        self.updates += 1
        if self.updates % 100 == 0:
            self.target.load_state_dict(self.online.state_dict())
        return {"loss": float(loss.item()), "td_error": float(td_error.mean().item()), "beta": self.replay.beta}

In [ ]:
agent_demo = RainbowDQNAgent(obs_dim=4, n_actions=2, batch_size=32, capacity=500, v_min=0.0, v_max=500.0, n_atoms=51)
for _ in range(8):
    s = np.random.randn(4).astype(np.float32)
    ns = np.random.randn(4).astype(np.float32)
    agent_demo.store_transition(s, np.random.randint(2), 1.0, ns, False)
print("Action demo:", agent_demo.select_action(np.zeros(4, dtype=np.float32)))
print("Replay size:", len(agent_demo.replay))
print("Learn metrics keys:", list(agent_demo.learn_step().keys()))

## 9. Pseudocode Rainbow complet

Maintenant que chaque composant a un sens, on peut lire l'algorithme complet. Le point important est de ne pas voir Rainbow comme une boucle entierement nouvelle. C'est une boucle DQN dans laquelle chaque etape a ete enrichie.

$$
\boxed{
\begin{aligned}
&\textbf{Algorithme : Rainbow DQN} \\
&\textbf{Entrees : } Z_\theta,\ Z_{\theta^-},\ \mathcal{D}_{PER},\ \mathcal{N},\ \mathcal{Z}=\{z_j\}_{j=1}^{N_{atoms}} \\
&\textbf{Hyperparametres : } \gamma,\ n,\ \alpha_{PER},\ \beta,\ B,\ \eta,\ C,\ V_{min},\ V_{max} \\
&\textbf{Initialisation : } \theta^- \leftarrow \theta,\ \mathcal{D}_{PER}\leftarrow\emptyset,\ \mathcal{N}\leftarrow\emptyset \\
\\
&\textbf{Pour chaque episode :} \\
&\quad \text{initialiser } s_0 \text{ et vider l'accumulateur } n\text{-step } \mathcal{N} \\
&\quad \textbf{Tant que l'episode n'est pas termine :} \\
&\qquad 1.\; \text{resampler le bruit des couches NoisyLinear de } Z_\theta \\
&\qquad 2.\; Q_\theta(s_t,a) \leftarrow \sum_{j=1}^{N_{atoms}} z_j\,p_\theta(z_j\mid s_t,a) \\
&\qquad\quad a_t \leftarrow \arg\max_a Q_\theta(s_t,a) \\
&\qquad 3.\; \text{executer } a_t,\ \text{observer } (r_t,s_{t+1},d_t) \\
&\qquad 4.\; \text{ajouter } (s_t,a_t,r_t,s_{t+1},d_t) \text{ dans } \mathcal{N} \\
&\qquad 5.\; \text{si } (x_i,a_i,R_i^{(n)},x'_i,d_i) \text{ est pret, l'inserer dans } \mathcal{D}_{PER} \\
\\
&\qquad 6.\; \textbf{Si } |\mathcal{D}_{PER}| \ge B : \\
&\qquad\quad P(i)=\frac{p_i^{\alpha_{PER}}}{\sum_k p_k^{\alpha_{PER}}},\qquad
w_i=\left(\frac{1}{|\mathcal{D}_{PER}|P(i)}\right)^\beta \Big/ \max_k w_k \\
&\qquad\quad \text{echantillonner } \{(x_i,a_i,R_i^{(n)},x'_i,d_i,w_i)\}_{i=1}^{B} \sim P(i) \\
&\qquad\quad \text{resampler le bruit de } Z_\theta \text{ et } Z_{\theta^-} \\
&\qquad\quad a_i^* \leftarrow \arg\max_a \sum_j z_j\,p_\theta(z_j\mid x'_i,a) \qquad \text{(selection online)} \\
&\qquad\quad \hat{p}_i \leftarrow p_{\theta^-}(\cdot\mid x'_i,a_i^*) \qquad\qquad\qquad\ \text{(evaluation target)} \\
&\qquad\quad Tz_i \leftarrow \mathrm{clip}\left(R_i^{(n)}+(1-d_i)\gamma^n z,\ V_{min},\ V_{max}\right) \\
&\qquad\quad m_i \leftarrow \Pi_{\mathcal{Z}}(Tz_i,\hat{p}_i) \qquad \text{(projection C51)} \\
&\qquad\quad p_i^{online} \leftarrow p_\theta(\cdot\mid x_i,a_i) \\
&\qquad\quad \tilde{\ell}_i \leftarrow -\sum_{j=1}^{N_{atoms}} m_i^{(j)}\log p_{i,online}^{(j)} \\
&\qquad\quad \mathcal{L}(\theta) \leftarrow \frac{1}{B}\sum_{i=1}^{B} w_i\,\tilde{\ell}_i \\
&\qquad\quad \theta \leftarrow \theta - \eta\nabla_\theta\mathcal{L}(\theta) \\
&\qquad\quad p_i \leftarrow \tilde{\ell}_i + \varepsilon_{PER} \qquad \text{mettre a jour les priorites} \\
\\
&\qquad 7.\; \text{toutes les } C \text{ mises a jour : } \theta^- \leftarrow \theta \\
&\qquad 8.\; s_t \leftarrow s_{t+1}
\end{aligned}
}
$$

### Comment lire ce pseudocode

Les lignes 2, 3, 6 et 7 sont le squelette **DQN** : choisir une action discrete, interagir avec l'environnement, apprendre depuis un replay buffer, puis copier periodiquement le reseau online vers le target network.

Les lignes $a_i^* = argmax ... online$ puis $p_{\theta^-}(..., a_i^*)$ sont la partie **Double DQN**. Le reseau online choisit l'action suivante, le target network l'evalue. Comme C51 predit une distribution, le $argmax$ se fait sur l'esperance de cette distribution :

$$Q(s,a)=\mathbb{E}[Z(s,a)] = \sum_j z_j p(z_j\mid s,a).$$

Les lignes avec $P(i)$ et $w_i$ sont **Prioritized Experience Replay** : le batch est biaise vers les transitions prioritaires, puis la loss est corrigee par les poids d'importance sampling.

Les lignes avec $R_i^{(n)}$ et $\gamma^n$ sont les **n-step returns** : la transition stockee contient deja plusieurs rewards accumulees avant le bootstrap.

Les lignes $Tz_i$, $\Pi_Z$ et la cross-entropy sont **C51**. La cible n'est plus un scalaire, mais une distribution projetee sur le support fixe. C'est pour cela que la loss est une cross-entropy entre deux distributions, et non une MSE entre deux valeurs.

Enfin, le resampling des `NoisyLinear` intervient avant la selection d'action et pendant l'update. Rainbow reste greedy, mais greedy par rapport a une version bruit ee du reseau pendant l'entrainement.


## 10. Training optionnel sur CartPole-v1

Le training reste optionnel : le flag ci-dessous permet de le désactiver lors d'une relecture rapide. Il sert de demonstration, pas de benchmark.

Quand on active `RUN_RAINBOW_TRAINING`, il faut lire les courbes avec prudence : CartPole est simple, Rainbow est surdimensionne, et les runs courts peuvent varier selon la seed. Le but est de verifier que la boucle fonctionne et que les diagnostics ont du sens.

Ce qu'on surveille :

| Courbe | Lecture utile |
|---|---|
| reward episode | l'agent survit-il plus longtemps, ou bien quelques episodes isoles tirent-ils la moyenne ? |
| loss distributionnelle | du bruit est normal; une derive durable ou des pics qui ne retombent pas signalent une cible mal alignee |
| TD error | il baisse rarement de facon monotone, mais il doit arreter d'exploser quand le replay commence a couvrir des situations deja vues |
| beta PER | sa hausse progressive indique que la correction du biais du replay devient plus stricte au fil des updates |
| eval reward | c'est le meilleur thermometre: politique greedy, sans bruit de NoisyNet ni variance d'episode individuelle |

La bonne lecture consiste a croiser ces panneaux. Par exemple, une reward d'entrainement qui monte pendant que l'eval stagne peut simplement vouloir dire que le bruit d'exploration aide encore a survivre; a l'inverse, une eval qui grimpe alors que la loss reste chaotique est acceptable tant que la masse distributionnelle et le TD error ne se desorganisent pas durablement.

In [ ]:
# Configuration du run Rainbow optionnel.
SEED = 42
ENV_ID = "CartPole-v1"
RUN_RAINBOW_TRAINING = True
TOTAL_EPISODES = 150
EVAL_EVERY = 20
MAX_STEPS = 500

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
def evaluate_agent(agent, n_episodes=5, seed=10_000):
    env = gym.make(ENV_ID)
    rewards = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total = 0.0
        for _ in range(MAX_STEPS):
            action = agent.select_action(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            total += reward
            if terminated or truncated:
                break
        rewards.append(total)
    env.close()
    return float(np.mean(rewards)), float(np.std(rewards))


if RUN_RAINBOW_TRAINING:
    env = gym.make(ENV_ID)
    env.action_space.seed(SEED)
    agent = RainbowDQNAgent(obs_dim=4, n_actions=2, batch_size=32, capacity=5000)
    history = {"episode_rewards": [], "losses": [], "td_errors": [], "betas": [], "eval_steps": [], "eval_means": [], "eval_stds": []}

    for episode in range(1, TOTAL_EPISODES + 1):
        obs, _ = env.reset(seed=SEED + episode)
        ep_reward = 0.0
        for _ in range(MAX_STEPS):
            action = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.store_transition(obs, action, reward, next_obs, done)
            metrics = agent.learn_step()
            if metrics:
                history["losses"].append(metrics["loss"])
                history["td_errors"].append(metrics["td_error"])
                history["betas"].append(metrics["beta"])
            obs = next_obs
            ep_reward += reward
            if done:
                break
        for transition in agent.n_step.flush():
            agent.replay.push(*transition)
        history["episode_rewards"].append(ep_reward)
        if episode % EVAL_EVERY == 0:
            mean, std = evaluate_agent(agent)
            history["eval_steps"].append(episode)
            history["eval_means"].append(mean)
            history["eval_stds"].append(std)
            print(f"Episode {episode:03d} | train reward={ep_reward:.1f} | eval={mean:.1f} +/- {std:.1f}")
    env.close()
else:
    history = None
    print("Training Rainbow desactive. Mettre RUN_RAINBOW_TRAINING = True pour generer les courbes.")


In [ ]:
if history is None:
    print("Aucune courbe a tracer : activer RUN_RAINBOW_TRAINING pour generer un historique.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    rewards = np.array(history["episode_rewards"], dtype=np.float32)
    axes[0, 0].plot(rewards, alpha=0.35, label="reward brute")
    if len(rewards) >= 10:
        smooth = np.convolve(rewards, np.ones(10) / 10, mode="valid")
        axes[0, 0].plot(np.arange(9, len(rewards)), smooth, linewidth=2, label="moy. glissante 10")
    axes[0, 0].set_title("Reward par episode")
    axes[0, 0].legend()

    axes[0, 1].plot(history["losses"], color="darkred")
    axes[0, 1].set_title("Loss distributionnelle")

    axes[0, 2].plot(history["td_errors"], color="purple")
    axes[0, 2].set_title("TD error moyen")

    axes[1, 0].plot(history["betas"], color="darkorange")
    axes[1, 0].set_title("PER beta")

    if history["eval_steps"]:
        axes[1, 1].errorbar(history["eval_steps"], history["eval_means"], yerr=history["eval_stds"], marker="o")
    axes[1, 1].set_title("Evaluation greedy")

    axes[1, 2].axis("off")
    axes[1, 2].text(
        0,
        0.92,
        "Lecture detaillee:\n"
        "- reward brute haute + eval basse : le bruit aide encore a survivre\n"
        "- loss bruitee seule : normal; loss + TD error qui s'envolent : instabilite\n"
        "- beta qui monte sans eval qui suit : PER corrige mieux, mais la politique n'exploite pas encore\n"
        "- eval moyenne + barre d'erreur : separer progression reelle et variance entre episodes",
        fontsize=10,
        va="top",
    )

    for ax in axes.ravel():
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Démonstration finale : replay vidéo de la politique apprise ===

def evaluate_and_display_agent(
    agent,
    label="Rainbow DQN",
    n_episodes=3,
    seed=20_000,
    video_root="videos/02b_rainbow_dqn_cartpole",
):
    """Évalue une politique greedy et affiche le dernier replay vidéo enregistré."""
    rewards = []
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    try:
        for episode_idx in range(n_episodes):
            obs, _ = env.reset(seed=seed + episode_idx)
            total_reward = 0.0

            for _ in range(500):
                action = agent.select_action(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward

                if terminated or truncated:
                    break

            rewards.append(total_reward)
            print(f"[{label}] Épisode {episode_idx + 1} : {total_reward:.0f} steps")

    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):.1f} steps\n")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return rewards


if history is None or "agent" not in globals():
    print("Aucun agent entraîné disponible : lance d'abord la cellule de training Rainbow.")
else:
    rainbow_demo = evaluate_and_display_agent(agent, "Rainbow DQN")

## Conclusion et limites

Rainbow assemble plusieurs idees qui corrigent chacune une faiblesse precise du DQN classique.

- **Double DQN** reduit l'overestimation en separant selection et evaluation.
- **Dueling** structure le reseau en distinguant valeur d'etat et avantage d'action.
- **PER** donne plus d'updates aux transitions que le reseau comprend mal.
- **n-step** propage plus vite les rewards vers les decisions passees.
- **C51** apprend une distribution de retour au lieu d'une moyenne unique.
- **Noisy Nets** remplacent l'exploration epsilon-greedy par une perturbation apprise des parametres.

La morale du notebook n'est pas que Rainbow est necessaire pour CartPole. Il ne l'est pas. La morale est que Rainbow montre comment un algorithme deep RL mature se construit souvent : non pas avec une seule astuce miraculeuse, mais avec un ensemble de corrections compatibles qui s'attaquent a des sources differentes d'instabilite.

Pour un vrai benchmark Rainbow, il faudrait passer a des environnements plus exigeants, utiliser un replay priorise plus efficace (sum-tree), lancer plusieurs seeds, et comparer contre DQN/Double DQN sur un budget d'interactions identique. Ici, on a construit le modele mental et l'implementation minimale pour comprendre ce que ferait cette version plus lourde.


## Sources primaires

- Hessel, M. et al. (2018). *Rainbow: Combining Improvements in Deep Reinforcement Learning*. [arXiv:1710.02298](https://arxiv.org/abs/1710.02298).
- van Hasselt, H., Guez, A. & Silver, D. (2016). *Deep Reinforcement Learning with Double Q-learning*. [arXiv:1509.06461](https://arxiv.org/abs/1509.06461).
- Wang, Z. et al. (2016). *Dueling Network Architectures for Deep Reinforcement Learning*. [arXiv:1511.06581](https://arxiv.org/abs/1511.06581).
- Schaul, T. et al. (2016). *Prioritized Experience Replay*. [arXiv:1511.05952](https://arxiv.org/abs/1511.05952).
- Bellemare, M. et al. (2017). *A Distributional Perspective on Reinforcement Learning*. [arXiv:1707.06887](https://arxiv.org/abs/1707.06887).
- Fortunato, M. et al. (2018). *Noisy Networks for Exploration*. [arXiv:1706.10295](https://arxiv.org/abs/1706.10295).